# Fase 5 - Segmentacion de Viajes con Clustering

**Pregunta de negocio:** ¿Existen segmentos naturales de viajes en NYC?

## Objetivos de este notebook

1. Cargar una muestra de ~100K viajes con features clave desde BigQuery
2. Escalar las features con StandardScaler para clustering
3. Aplicar K-Means con distintos valores de k (2-10)
4. Seleccionar el k optimo con el metodo del codo y el coeficiente de silueta
5. Perfilar los clusters y asignarles nombres descriptivos
6. Visualizar los clusters con PCA y radar charts
7. Guardar el modelo final para uso futuro

## ¿Por que segmentar viajes?

La segmentacion nos permite descubrir **patrones ocultos** en los datos. En lugar de tratar
todos los viajes como iguales, podemos identificar grupos con caracteristicas similares
(por ejemplo, viajes al aeropuerto vs. viajes urbanos cortos) y diseñar estrategias
especificas para cada segmento.

## 1. Configuracion e importaciones

In [ ]:
# Conexion a BigQuery
import sys
sys.path.insert(0, '../../../src')
from bigquery.bq_helper import BigQueryHelper
bq = BigQueryHelper()

In [ ]:
# Librerias de analisis y visualizacion
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.decomposition import PCA
import joblib
import os

# Configuracion de estilo
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# Semilla para reproducibilidad
RANDOM_STATE = 42

## 2. Carga de datos desde BigQuery

Extraemos ~100,000 viajes con las features que usaremos para el clustering:
- **distance**: distancia del viaje en millas
- **duration_min**: duracion del viaje en minutos
- **fare_amount**: tarifa en USD
- **tip_pct**: porcentaje de propina
- **speed_mph**: velocidad promedio en millas por hora
- **hour**: hora de recogida (0-23)
- **is_weekend**: indicador de fin de semana (0/1)

In [ ]:
query = """
SELECT
    trip_distance AS distance,
    TIMESTAMP_DIFF(dropoff_datetime, pickup_datetime, SECOND) / 60.0 AS duration_min,
    fare_amount,
    CASE
        WHEN fare_amount > 0 THEN (tip_amount / fare_amount) * 100
        ELSE 0
    END AS tip_pct,
    CASE
        WHEN TIMESTAMP_DIFF(dropoff_datetime, pickup_datetime, SECOND) > 0
             AND trip_distance > 0
        THEN trip_distance / (TIMESTAMP_DIFF(dropoff_datetime, pickup_datetime, SECOND) / 3600.0)
        ELSE NULL
    END AS speed_mph,
    EXTRACT(HOUR FROM pickup_datetime) AS hour,
    CASE
        WHEN EXTRACT(DAYOFWEEK FROM pickup_datetime) IN (1, 7) THEN 1
        ELSE 0
    END AS is_weekend
FROM
    `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2015`
WHERE
    fare_amount > 0 AND fare_amount < 500
    AND trip_distance > 0 AND trip_distance < 100
    AND passenger_count > 0
    AND pickup_location_id IS NOT NULL
    AND TIMESTAMP_DIFF(dropoff_datetime, pickup_datetime, SECOND) > 60
    AND TIMESTAMP_DIFF(dropoff_datetime, pickup_datetime, SECOND) < 7200
ORDER BY RAND()
LIMIT 100000
"""

df = bq.query_to_df(query)
print(f"Registros cargados: {len(df):,}")
df.head()

In [ ]:
# Verificacion rapida
print("Tipos de datos:")
print(df.dtypes)
print(f"\nValores nulos por columna:")
print(df.isnull().sum())
print(f"\nEstadisticas descriptivas:")
df.describe().round(2)

In [ ]:
# Eliminar filas con valores nulos (necesario para clustering)
df_clean = df.dropna().copy()

# Filtrar velocidades extremas (posibles errores de calculo)
df_clean = df_clean[(df_clean['speed_mph'] > 0) & (df_clean['speed_mph'] < 80)]
df_clean = df_clean[df_clean['tip_pct'] < 100]  # propinas razonables

print(f"Registros despues de limpieza: {len(df_clean):,}")
print(f"Registros eliminados: {len(df) - len(df_clean):,}")

## 3. Feature Scaling con StandardScaler

K-Means calcula distancias euclidianas entre puntos, por lo que es **sensible a la escala**
de las variables. Si una variable tiene un rango mucho mayor que otra, dominara el calculo
de distancias.

**StandardScaler** transforma cada variable para que tenga media=0 y desviacion estandar=1:

$$z = \frac{x - \mu}{\sigma}$$

In [ ]:
# Definir las features para clustering
feature_cols = ['distance', 'duration_min', 'fare_amount', 'tip_pct', 'speed_mph', 'hour', 'is_weekend']
X = df_clean[feature_cols].values

# Escalar
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Verificar que el escalado funciono correctamente
print("Verificacion del escalado:")
print(f"{'Feature':>15s} | {'Media orig':>12s} | {'Std orig':>12s} | {'Media scaled':>12s} | {'Std scaled':>12s}")
print("-" * 75)
for i, col in enumerate(feature_cols):
    print(f"{col:>15s} | {X[:, i].mean():>12.2f} | {X[:, i].std():>12.2f} | {X_scaled[:, i].mean():>12.4f} | {X_scaled[:, i].std():>12.4f}")

## 4. K-Means: Probar k de 2 a 10

Ejecutamos K-Means para un rango de k (numero de clusters) y registramos:
- **Inercia (inertia):** suma de distancias al cuadrado de cada punto a su centroide
- **Silhouette score:** medida de que tan bien separados estan los clusters (-1 a 1)

In [ ]:
K_range = range(2, 11)
inertias = []
silhouette_scores = []

print("Entrenando K-Means para diferentes valores de k...")
print(f"{'k':>3s} | {'Inercia':>15s} | {'Silhouette':>12s}")
print("-" * 40)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10, max_iter=300)
    labels = kmeans.fit_predict(X_scaled)
    
    inertias.append(kmeans.inertia_)
    sil_score = silhouette_score(X_scaled, labels, sample_size=10000, random_state=RANDOM_STATE)
    silhouette_scores.append(sil_score)
    
    print(f"{k:>3d} | {kmeans.inertia_:>15,.0f} | {sil_score:>12.4f}")

print("\nEntrenamiento completado.")

## 5. Metodo del codo (Elbow Method)

Buscamos el punto donde la reduccion de la inercia se desacelera significativamente
("el codo" de la curva). Mas alla de ese punto, agregar clusters adicionales
aporta poca mejoria.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Grafico de inercia (metodo del codo)
axes[0].plot(list(K_range), inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Numero de clusters (k)')
axes[0].set_ylabel('Inercia')
axes[0].set_title('Metodo del Codo: Inercia vs k', fontweight='bold')
axes[0].set_xticks(list(K_range))

# Marcar el codo sugerido
axes[0].axvline(x=4, color='red', linestyle='--', alpha=0.7, label='Codo sugerido (k=4)')
axes[0].legend()

# Grafico de silhouette score
axes[1].plot(list(K_range), silhouette_scores, 'rs-', linewidth=2, markersize=8)
axes[1].set_xlabel('Numero de clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score vs k', fontweight='bold')
axes[1].set_xticks(list(K_range))

# Marcar el mejor silhouette
best_k_idx = np.argmax(silhouette_scores)
best_k = list(K_range)[best_k_idx]
axes[1].axvline(x=best_k, color='green', linestyle='--', alpha=0.7,
                label=f'Mejor silhouette (k={best_k})')
axes[1].legend()

plt.suptitle('Seleccion del Numero Optimo de Clusters', fontweight='bold', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print(f"\nMejor silhouette score: {silhouette_scores[best_k_idx]:.4f} con k={best_k}")

## 6. Silhouette Plot para el k optimo

El silhouette plot muestra el coeficiente de silueta de **cada observacion** individual,
agrupado por cluster. Esto nos permite:
- Ver si todos los clusters tienen un grosor similar (equilibrio)
- Detectar clusters con valores negativos (observaciones mal asignadas)
- Confirmar nuestra eleccion de k

In [ ]:
# Usar k=4 como optimo (ajustar segun los resultados de arriba)
OPTIMAL_K = 4

kmeans_final = KMeans(n_clusters=OPTIMAL_K, random_state=RANDOM_STATE, n_init=10, max_iter=300)
cluster_labels = kmeans_final.fit_predict(X_scaled)

# Calcular silhouette para cada muestra
sample_silhouette_values = silhouette_samples(X_scaled, cluster_labels)
avg_silhouette = silhouette_score(X_scaled, cluster_labels)

fig, ax = plt.subplots(figsize=(12, 8))

y_lower = 10
colors = cm.Set2(np.linspace(0, 1, OPTIMAL_K))

for i in range(OPTIMAL_K):
    # Valores de silueta del cluster i, ordenados
    ith_cluster_values = sample_silhouette_values[cluster_labels == i]
    ith_cluster_values.sort()
    
    size_cluster_i = ith_cluster_values.shape[0]
    y_upper = y_lower + size_cluster_i
    
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, ith_cluster_values,
                     facecolor=colors[i], edgecolor=colors[i], alpha=0.7)
    
    # Etiqueta del cluster
    ax.text(-0.05, y_lower + 0.5 * size_cluster_i, f'Cluster {i}',
            fontweight='bold', fontsize=11)
    
    y_lower = y_upper + 10

ax.set_title(f'Silhouette Plot para k={OPTIMAL_K} clusters', fontweight='bold')
ax.set_xlabel('Coeficiente de Silueta')
ax.set_ylabel('Cluster')

# Linea vertical con el promedio
ax.axvline(x=avg_silhouette, color='red', linestyle='--', linewidth=2,
           label=f'Promedio: {avg_silhouette:.3f}')
ax.legend(fontsize=12)
ax.set_yticks([])

plt.tight_layout()
plt.show()

print(f"\nSilhouette score promedio: {avg_silhouette:.4f}")
print(f"Tamanio de cada cluster:")
for i in range(OPTIMAL_K):
    n = (cluster_labels == i).sum()
    print(f"  Cluster {i}: {n:,} viajes ({n/len(cluster_labels)*100:.1f}%)")

## 7. Perfil de los clusters

Calculamos la media y mediana de cada feature por cluster para entender
las caracteristicas que definen cada segmento.

In [ ]:
# Agregar etiquetas de cluster al dataframe
df_clean = df_clean.copy()
df_clean['cluster'] = cluster_labels

# Media por cluster
print("MEDIA de cada feature por cluster:")
print("=" * 80)
cluster_means = df_clean.groupby('cluster')[feature_cols].mean().round(2)
display(cluster_means)

print("\nMEDIANA de cada feature por cluster:")
print("=" * 80)
cluster_medians = df_clean.groupby('cluster')[feature_cols].median().round(2)
display(cluster_medians)

In [ ]:
# Visualizar los perfiles con heatmap
fig, ax = plt.subplots(figsize=(12, 5))

# Normalizar las medias para el heatmap (z-score por columna)
cluster_means_norm = (cluster_means - cluster_means.mean()) / cluster_means.std()

sns.heatmap(cluster_means_norm.astype(float), annot=cluster_means.values, fmt='.1f',            cmap='RdYlGn', center=0, linewidths=1, ax=ax,
            xticklabels=feature_cols,
            yticklabels=[f'Cluster {i}' for i in range(OPTIMAL_K)])

ax.set_title('Perfil de Clusters (valores = media, color = z-score relativo)', fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Nombrar los clusters segun sus perfiles

Basandonos en los perfiles de media y mediana, asignamos nombres descriptivos
a cada cluster. Los nombres tipicos que esperamos encontrar:

- **Commuter (pendular):** viajes de distancia/duracion media, en horario laboral
- **Airport (aeropuerto):** viajes largos, tarifas altas, alta velocidad
- **Nocturno:** viajes en horas de la noche/madrugada, posiblemente fin de semana
- **Urbano corto:** viajes cortos, baja tarifa, baja velocidad (trafico)

In [ ]:
# Asignacion de nombres (ajustar segun los resultados reales de los perfiles)
# Los indices de los clusters pueden variar; revisar cluster_means arriba
# y asignar los nombres que correspondan.

# Identificar clusters por sus caracteristicas dominantes
cluster_names = {}

# Cluster con mayor distancia promedio -> airport
airport_cluster = cluster_means['distance'].idxmax()
cluster_names[airport_cluster] = 'airport'

# Cluster con mayor hora promedio (noche) -> nocturno
remaining = [c for c in range(OPTIMAL_K) if c not in cluster_names]
nocturno_cluster = cluster_means.loc[remaining, 'hour'].idxmax()
cluster_names[nocturno_cluster] = 'nocturno'

# Cluster con menor distancia -> urbano_corto
remaining = [c for c in range(OPTIMAL_K) if c not in cluster_names]
urbano_cluster = cluster_means.loc[remaining, 'distance'].idxmin()
cluster_names[urbano_cluster] = 'urbano_corto'

# El restante -> commuter
remaining = [c for c in range(OPTIMAL_K) if c not in cluster_names]
for c in remaining:
    cluster_names[c] = 'commuter'

df_clean['cluster_name'] = df_clean['cluster'].map(cluster_names)

print("Asignacion de nombres:")
for k, v in sorted(cluster_names.items()):
    n = (df_clean['cluster'] == k).sum()
    print(f"  Cluster {k} -> {v:15s} ({n:,} viajes)")

print("\nPerfil resumido por nombre de cluster:")
df_clean.groupby('cluster_name')[feature_cols].mean().round(2)

## 9. Visualizacion PCA 2D

Proyectamos los datos de 7 dimensiones a 2 dimensiones usando PCA (Analisis de
Componentes Principales) para poder visualizar los clusters en un scatter plot.

In [ ]:
# Reduccion dimensional con PCA
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

print(f"Varianza explicada por PC1: {pca.explained_variance_ratio_[0]:.2%}")
print(f"Varianza explicada por PC2: {pca.explained_variance_ratio_[1]:.2%}")
print(f"Varianza total explicada:   {sum(pca.explained_variance_ratio_):.2%}")

# Scatter plot
fig, ax = plt.subplots(figsize=(12, 9))

colors_map = {'commuter': '#2196F3', 'airport': '#FF5722',
              'nocturno': '#9C27B0', 'urbano_corto': '#4CAF50'}

# Tomar una submuestra para el scatter (evitar sobreploteo)
sample_idx = np.random.RandomState(RANDOM_STATE).choice(len(X_pca), size=min(10000, len(X_pca)), replace=False)

for name in colors_map:
    mask = df_clean.iloc[sample_idx]['cluster_name'].values == name
    ax.scatter(X_pca[sample_idx][mask, 0], X_pca[sample_idx][mask, 1],
               c=colors_map[name], label=name, alpha=0.4, s=15, edgecolors='none')

# Plotear centroides
centroids_pca = pca.transform(kmeans_final.cluster_centers_)
for i, name in cluster_names.items():
    ax.scatter(centroids_pca[i, 0], centroids_pca[i, 1],
               c='black', marker='X', s=200, edgecolors='white', linewidths=2,
               zorder=5)
    ax.annotate(name, (centroids_pca[i, 0], centroids_pca[i, 1]),
                fontsize=11, fontweight='bold',
                xytext=(10, 10), textcoords='offset points',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.8))

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} varianza)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} varianza)')
ax.set_title('Clusters de Viajes - Proyeccion PCA 2D', fontweight='bold', fontsize=14)
ax.legend(title='Segmento', fontsize=11, title_fontsize=12, markerscale=3)

plt.tight_layout()
plt.show()

## 10. Radar Charts comparativos

Los radar charts (o spider charts) son ideales para comparar perfiles multidimensionales.
Cada eje del radar representa una feature, y cada cluster se dibuja como un poligono.

In [ ]:
# Preparar datos para radar chart (normalizar al rango 0-1)
from sklearn.preprocessing import MinMaxScaler

radar_data = df_clean.groupby('cluster_name')[feature_cols].mean()
radar_normalized = pd.DataFrame(
    MinMaxScaler().fit_transform(radar_data),
    index=radar_data.index,
    columns=radar_data.columns
)

# Crear radar chart
categories = feature_cols
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]  # cerrar el poligono

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))

for cluster_name in radar_normalized.index:
    values = radar_normalized.loc[cluster_name].values.flatten().tolist()
    values += values[:1]  # cerrar el poligono
    
    color = colors_map.get(cluster_name, '#999999')
    ax.plot(angles, values, 'o-', linewidth=2, label=cluster_name, color=color)
    ax.fill(angles, values, alpha=0.15, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=11)
ax.set_ylim(0, 1)
ax.set_title('Perfil Comparativo de Clusters (Radar Chart)', fontweight='bold',
             fontsize=14, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=11)

plt.tight_layout()
plt.show()

print("Interpretacion:")
print("- Cada eje representa una feature normalizada al rango [0, 1]")
print("- Un valor mas cercano al borde exterior indica un valor mayor relativo")
print("- La forma del poligono define el 'perfil' de cada segmento")

## 11. Tabulacion cruzada: cluster x zona y cluster x banda horaria

Exploramos como se distribuyen los clusters en funcion de la zona de recogida
y la banda horaria del dia.

In [ ]:
# Crear bandas horarias
def hour_band(h):
    if 6 <= h < 10:
        return 'Maniana (6-10)'
    elif 10 <= h < 16:
        return 'Mediodia (10-16)'
    elif 16 <= h < 20:
        return 'Tarde (16-20)'
    elif 20 <= h < 24:
        return 'Noche (20-24)'
    else:
        return 'Madrugada (0-6)'

df_clean['hour_band'] = df_clean['hour'].apply(hour_band)

# Tabulacion cruzada: cluster x banda horaria
ct_hour = pd.crosstab(df_clean['cluster_name'], df_clean['hour_band'], normalize='index')
ct_hour = ct_hour[['Madrugada (0-6)', 'Maniana (6-10)', 'Mediodia (10-16)',
                    'Tarde (16-20)', 'Noche (20-24)']]

print("Distribucion porcentual de cada cluster por banda horaria:")
print("=" * 70)
display((ct_hour * 100).round(1))

# Visualizar
fig, ax = plt.subplots(figsize=(12, 6))
ct_hour.plot(kind='bar', stacked=True, ax=ax, colormap='viridis', edgecolor='white')
ax.set_title('Distribucion de Clusters por Banda Horaria', fontweight='bold')
ax.set_xlabel('Segmento de Viaje')
ax.set_ylabel('Proporcion')
ax.legend(title='Banda Horaria', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Tabulacion cruzada: cluster x dia de la semana
ct_weekend = pd.crosstab(df_clean['cluster_name'], df_clean['is_weekend'],
                         normalize='index')
ct_weekend.columns = ['Dia laboral', 'Fin de semana']

print("Distribucion porcentual de cada cluster por tipo de dia:")
print("=" * 50)
display((ct_weekend * 100).round(1))

fig, ax = plt.subplots(figsize=(10, 5))
ct_weekend.plot(kind='bar', ax=ax, color=['#2196F3', '#FF9800'], edgecolor='white')
ax.set_title('Distribucion de Clusters: Dia Laboral vs Fin de Semana', fontweight='bold')
ax.set_xlabel('Segmento de Viaje')
ax.set_ylabel('Proporcion')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title='Tipo de Dia')
plt.tight_layout()
plt.show()

## 12. Guardar el modelo de clustering

Guardamos el modelo K-Means entrenado y el scaler para poder reutilizarlos
en notebooks posteriores o en produccion.

In [ ]:
# Crear directorio si no existe
model_dir = '../../../models/nyc_taxi'
os.makedirs(model_dir, exist_ok=True)

# Guardar modelo y scaler
model_path = os.path.join(model_dir, 'trip_clusters.joblib')
artifacts = {
    'kmeans_model': kmeans_final,
    'scaler': scaler,
    'feature_cols': feature_cols,
    'cluster_names': cluster_names,
    'optimal_k': OPTIMAL_K,
    'silhouette_score': avg_silhouette
}

joblib.dump(artifacts, model_path)
print(f"Modelo guardado en: {model_path}")
print(f"Tamanio del archivo: {os.path.getsize(model_path) / 1024:.1f} KB")

# Verificar que se puede cargar
loaded = joblib.load(model_path)
print(f"\nVerificacion de carga exitosa:")
print(f"  - Modelo: {type(loaded['kmeans_model']).__name__}")
print(f"  - k optimo: {loaded['optimal_k']}")
print(f"  - Silhouette: {loaded['silhouette_score']:.4f}")
print(f"  - Clusters: {loaded['cluster_names']}")

## Conclusiones

1. **Segmentos identificados:** Encontramos ~4 segmentos naturales de viajes en NYC que se diferencian por distancia, duracion, tarifa, velocidad y patron temporal.

2. **Perfiles claros:** Cada cluster tiene un perfil distintivo:
   - **Airport:** viajes largos, tarifas altas, velocidad alta (autopista)
   - **Commuter:** viajes de distancia media, horario laboral, propina moderada
   - **Nocturno:** viajes en horas de la noche y madrugada, fin de semana
   - **Urbano corto:** viajes cortos con trafico, baja velocidad y tarifa

3. **Calidad del clustering:** El silhouette score indica una separacion razonable entre clusters, aunque existe superposicion (natural en datos reales).

4. **Aplicaciones practicas:** Esta segmentacion puede usarse para:
   - Estrategias de precios diferenciadas
   - Asignacion de vehiculos segun tipo de demanda
   - Prediccion de propinas por segmento

### Proximos pasos
- Notebook 02: Clustering geografico con DBSCAN
- Notebook 03: Descomposicion de series temporales